序列数据采样：
- 随机采样（Random Sampling）：随机选择训练样本，样本之间不一定保持原始顺序。
- 顺序采样（Sequential Sampling）：按照原始数据的顺序依次生成训练样本，保持序列的连续性。

In [60]:
import numpy as np
import torch


In [4]:
corpus = np.arange(0, 33, step=1)
print(corpus)

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32]


In [77]:
# 1、随机采样（Random Sampling）
# 随机采样就是从原始序列中随机选择起始位置，截取固定长度的片段作为训练样本。
def random_sampling(token_ids, batch_size, num_steps):
    """
    随机采样序列数据
    :param token_ids: 原始 Token ID 序列
    :param batch_size: 每个 Batch 的样本数量
    :param num_steps: 每个样本的序列长度
    :return: list[list[int]] 所有采样数据
    """
    # -1 的原因是要保证样本的最后一个token有预测值
    num_samples = (len(token_ids) - 1) // num_steps
    # 随机一下初始位置
    init_index = np.random.randint(0, num_steps)
    # 获取所有样本的起始索引位置
    init_indices = np.arange(init_index, init_index + num_samples * num_steps, num_steps)
    # 打乱索引开始位置，以确保随机采样
    np.random.shuffle(init_indices)
    # 构造每个样本的所有索引
    def get_indices(token_indices, num_steps, start_index):
        return token_indices[start_index : start_index + num_steps]
    # 构造 batch
    res_X, res_Y = [], []
    for i in range(len(init_indices) // batch_size):
        per_batch_start_index = init_indices[i*batch_size: i*batch_size + batch_size]
        res_X.append([ get_indices(token_ids, num_steps, sindex) for sindex in per_batch_start_index])
        res_Y.append([ get_indices(token_ids, num_steps, sindex + 1) for sindex in per_batch_start_index])
    return torch.tensor(res_X), torch.tensor(res_Y)

rand_sample_X, rand_sample_Y = random_sampling(corpus, 2, 5)
print(f"rand_sample_X: {rand_sample_X}")
print(f"rand_sample_Y: {rand_sample_Y}")


init_indices :  [22 12  2 17 27  7]
per_batch_start_index :  [22 12]
per_batch_start_index :  [ 2 17]
per_batch_start_index :  [27  7]
rand_sample_X: tensor([[[22, 23, 24, 25, 26],
         [12, 13, 14, 15, 16]],

        [[ 2,  3,  4,  5,  6],
         [17, 18, 19, 20, 21]],

        [[27, 28, 29, 30, 31],
         [ 7,  8,  9, 10, 11]]])
rand_sample_Y: tensor([[[23, 24, 25, 26, 27],
         [13, 14, 15, 16, 17]],

        [[ 3,  4,  5,  6,  7],
         [18, 19, 20, 21, 22]],

        [[28, 29, 30, 31, 32],
         [ 8,  9, 10, 11, 12]]])


In [78]:
# 2、顺序采样（Sequential Sampling）
# 顺序采样是按照原始序列的顺序组织训练数据，使不同 Batch 之间尽可能保持连续性。
# 随机采样：每次随机选择一个序列片段。
# 顺序采样：按照原始序列的排列顺序，连续地构造训练片段。

def sequential_sampling(token_ids, batch_size, num_steps):
    """
    随机采样序列数据
    :param token_ids: 原始 Token ID 序列
    :param batch_size: 每个 Batch 的样本数量
    :param num_steps: 每个样本的序列长度
    :return: list[list[int]] 所有采样数据
    """
    # -1 的原因是要保证样本的最后一个token有预测值
    num_samples = (len(token_ids) - 1) // num_steps
    # 随机一下初始位置
    init_index = np.random.randint(0, num_steps)
    # 获取所有样本的起始索引位置
    init_indices = np.arange(init_index, init_index + num_samples * num_steps, num_steps)
    # 构造每个样本的所有索引
    def get_indices(token_indices, num_steps, start_index):
        return token_indices[start_index : start_index + num_steps]
    # 构造 batch
    res_X, res_Y = [], []
    for i in range(len(init_indices) // batch_size):
        per_batch_start_index = init_indices[i*batch_size: i*batch_size + batch_size]
        res_X.append([ get_indices(token_ids, num_steps, sindex) for sindex in per_batch_start_index])
        res_Y.append([ get_indices(token_ids, num_steps, sindex + 1) for sindex in per_batch_start_index])
    return torch.tensor(res_X), torch.tensor(res_Y)

rand_sample_X, rand_sample_Y = sequential_sampling(corpus, 2, 5)
print(f"rand_sample_X: {rand_sample_X}")
print(f"rand_sample_Y: {rand_sample_Y}")

rand_sample_X: tensor([[[ 0,  1,  2,  3,  4],
         [ 5,  6,  7,  8,  9]],

        [[10, 11, 12, 13, 14],
         [15, 16, 17, 18, 19]],

        [[20, 21, 22, 23, 24],
         [25, 26, 27, 28, 29]]])
rand_sample_Y: tensor([[[ 1,  2,  3,  4,  5],
         [ 6,  7,  8,  9, 10]],

        [[11, 12, 13, 14, 15],
         [16, 17, 18, 19, 20]],

        [[21, 22, 23, 24, 25],
         [26, 27, 28, 29, 30]]])
